# Lab 2 - Kernel Tuning

**ROCm Certification Program — Level 1**


<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">

<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">
🎯 Objectives
</div>

<p>
Starting from a deliberately suboptimal matrix multiplication kernel, improve its performance through a series of optimization steps.
</p>

<ul style="margin-bottom:0;">

<li>Measure kernel performance after each optimization</li>

<li>Improve global memory access patterns</li>

<li>Use LDS (shared memory) tiling to reduce global memory traffic</li>

<li>Understand how each optimization changes performance</li>

</ul>

</div>

## Background — Matrix Multiplication

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">The operation</div>
<p>Multiply <code>A</code> (M×K) by <code>B</code> (K×N) to produce <code>C</code> (M×N). Matrices are stored <b>row-major</b>, so correct indexing is:</p>
</div>

```text
A[row * K + k]
B[k   * N + col]
C[row * N + col]
```

```text
for row in M:
    for col in N:
        sum = 0
        for k in K:
            sum += A[row*K + k] * B[k*N + col]
        C[row*N + col] = sum
```

## Baseline — Naive Matrix Multiply

<div style="background:#fff7ea; border-left:5px solid #e0a020; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#8a5a06; font-size:1.05em; margin-bottom:8px;">⚠️ Intentionally non-optimal (for teaching)</div>
<ul style='margin-bottom:0;'><li>Poor / uncoalesced memory access</li><li>No tiling</li><li>No shared-memory (LDS) usage</li></ul>
</div>

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Timing GPU kernels with HIP events</div>
<p>We do not use <code>std::chrono</code> for GPU kernels. Kernels run asynchronously, so CPU wall-clock timers can be misleading. Use HIP events instead:</p>
</div>

```cpp
hipEventRecord(start);
kernel<<<...>>>();
hipEventRecord(stop);
hipEventSynchronize(stop);
hipEventElapsedTime(&ms, start, stop);
```

In [ ]:
%%writefile matmul.cpp
#include <hip/hip_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>


// ============================================================
// BASELINE: Naive Matrix Multiply
//
// Deliberately non-optimized implementation.
//
// Performance issues:
//   - Poor thread-to-data mapping
//   - No tiling
//   - No shared memory (LDS) usage
//
// Matrix sizes:
//
//   A : M x K
//   B : K x N
//   C : M x N
//
// ============================================================

__global__ void matmul_naive(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    // Deliberately poor thread mapping:
    // neighboring threads do not access neighboring
    // output elements efficiently.

    int col = blockIdx.y * blockDim.y + threadIdx.y;
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N)
    {
        float sum = 0.0f;

        for (int k = 0; k < K; k++)
        {
            sum +=
                A[row * K + k] *
                B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}


// ============================================================
// CPU reference implementation
// Used for correctness validation
// ============================================================

void matmul_cpu(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    for (int row = 0; row < M; row++)
    {
        for (int col = 0; col < N; col++)
        {
            float sum = 0.0f;

            for (int k = 0; k < K; k++)
            {
                sum +=
                    A[row * K + k] *
                    B[k * N + col];
            }

            C[row * N + col] = sum;
        }
    }
}


int main(int argc, char** argv)
{
    int M = 1024;
    int N = 1024;
    int K = 1024;

    size_t sA = M * K * sizeof(float);
    size_t sB = K * N * sizeof(float);
    size_t sC = M * N * sizeof(float);

    // Host memory

    float* h_A   = (float*)malloc(sA);
    float* h_B   = (float*)malloc(sB);
    float* h_C   = (float*)malloc(sC);
    float* h_ref = (float*)malloc(sC);

    // Initialize matrices with random values

    srand(42);

    for (int i = 0; i < M * K; i++)
        h_A[i] = (float)rand() / RAND_MAX;

    for (int i = 0; i < K * N; i++)
        h_B[i] = (float)rand() / RAND_MAX;

    // Device memory

    float* d_A;
    float* d_B;
    float* d_C;

    hipMalloc(&d_A, sA);
    hipMalloc(&d_B, sB);
    hipMalloc(&d_C, sC);

    hipMemcpy(d_A, h_A, sA, hipMemcpyHostToDevice);
    hipMemcpy(d_B, h_B, sB, hipMemcpyHostToDevice);

    dim3 block(16, 16); // Use 256 threads

    dim3 grid(
        (N + block.x - 1) / block.x,
        (M + block.y - 1) / block.y);

    // Warmup launch

    matmul_naive<<<grid, block>>>(
        d_A, d_B, d_C,
        M, N, K);

    hipDeviceSynchronize();

    // GPU timing using HIP events

    hipEvent_t start, stop;

    hipEventCreate(&start);
    hipEventCreate(&stop);

    hipEventRecord(start);

    for (int i = 0; i < 10; i++)
    {
        matmul_naive<<<grid, block>>>(
            d_A, d_B, d_C,
            M, N, K);
    }

    hipEventRecord(stop);
    hipEventSynchronize(stop);

    float ms = 0.0f;

    hipEventElapsedTime(&ms, start, stop);

    ms /= 10.0f;

    // Copy result back to CPU

    hipMemcpy(
        h_C,
        d_C,
        sC,
        hipMemcpyDeviceToHost);

    // Correctness validation

    matmul_cpu(
        h_A,
        h_B,
        h_ref,
        M, N, K);

    float maxErr = 0.0f;

    for (int i = 0; i < M * N; i++)
    {
        maxErr = fmax(
            maxErr,
            fabs(h_C[i] - h_ref[i]));
    }

    // GEMM performs:
    //
    //   2 * M * N * K floating-point operations

    double gflops =
        2.0 * M * N * K /
        (ms * 1e-3) /
        1e9;

    // Results

    printf("Kernel:    BASELINE (naive)\n");

    printf(
        "Block:     %dx%d = %d threads\n",
        block.x,
        block.y,
        block.x * block.y);

    printf("Time:      %.3f ms\n", ms);

    printf("GFLOPS:    %.1f\n", gflops);

    printf("Max error: %e\n", maxErr);

    printf(
        "Status:    %s\n",
        maxErr < 1e-2 ? "PASS" : "FAIL");

    // Cleanup

    hipFree(d_A);
    hipFree(d_B);
    hipFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);
    free(h_ref);

    return 0;
}

In [ ]:
# Compile and run baseline
import subprocess

subprocess.run(['hipcc', '-O3', '-o', 'matmul', 'matmul.cpp'],
               capture_output=True, text=True, check=True)
result = subprocess.run(['./matmul'], capture_output=True, text=True)
print("=== BASELINE RESULT ===")
print(result.stdout)

## Iteration 1 — Coalescing & Workgroup Configuration

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p>Improve memory coalescing and wavefront utilization so neighboring threads access neighboring memory. Swap the row/col mapping and grow the workgroup from 32 to 256 threads so the GPU runs multiple full wavefronts efficiently.</p>
</div>

```text
Baseline mapping                         Improved mapping
int col = blockIdx.y*blockDim.y + ty;    int row = blockIdx.y*blockDim.y + ty;
int row = blockIdx.x*blockDim.x + tx;    int col = blockIdx.x*blockDim.x + tx;
dim3 block(16, 16);   // 256 threads     dim3 block(16, 16); // 256 threads
```

In [ ]:
%%writefile matmul_v1.cpp
#include <hip/hip_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>


// ============================================================
// Iteration 1:
// Improve coalescing and workgroup configuration
//
// Changes from baseline:
//   - Better thread-to-data mapping
//   - Better memory coalescing
//   - Larger workgroup: 32 -> 256 threads
//
// Still missing:
//   - LDS / shared memory tiling
// ============================================================

__global__ void matmul_coalesced(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    // FIXED mapping:
    // neighboring threads now process neighboring columns

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N)
    {
        float sum = 0.0f;

        for (int k = 0; k < K; k++)
            sum += A[row * K + k] * B[k * N + col];

        C[row * N + col] = sum;
    }
}


// CPU reference implementation

void matmul_cpu(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    for (int row = 0; row < M; row++)
    {
        for (int col = 0; col < N; col++)
        {
            float sum = 0.0f;

            for (int k = 0; k < K; k++)
                sum += A[row * K + k] * B[k * N + col];

            C[row * N + col] = sum;
        }
    }
}


int main()
{
    int M = 1024;
    int N = 1024;
    int K = 1024;

    size_t sA = M * K * sizeof(float);
    size_t sB = K * N * sizeof(float);
    size_t sC = M * N * sizeof(float);

    float* h_A   = (float*)malloc(sA);
    float* h_B   = (float*)malloc(sB);
    float* h_C   = (float*)malloc(sC);
    float* h_ref = (float*)malloc(sC);

    srand(42);

    for (int i = 0; i < M * K; i++)
        h_A[i] = (float)rand() / RAND_MAX;

    for (int i = 0; i < K * N; i++)
        h_B[i] = (float)rand() / RAND_MAX;

    float *d_A, *d_B, *d_C;

    hipMalloc(&d_A, sA);
    hipMalloc(&d_B, sB);
    hipMalloc(&d_C, sC);

    hipMemcpy(d_A, h_A, sA, hipMemcpyHostToDevice);
    hipMemcpy(d_B, h_B, sB, hipMemcpyHostToDevice);

    dim3 block(16, 16);

    dim3 grid(
        (N + block.x - 1) / block.x,
        (M + block.y - 1) / block.y);

    // Warmup

    matmul_coalesced<<<grid, block>>>(
        d_A, d_B, d_C,
        M, N, K);

    hipDeviceSynchronize();

    // GPU timing

    hipEvent_t start, stop;

    hipEventCreate(&start);
    hipEventCreate(&stop);

    hipEventRecord(start);

    for (int i = 0; i < 10; i++)
    {
        matmul_coalesced<<<grid, block>>>(
            d_A, d_B, d_C,
            M, N, K);
    }

    hipEventRecord(stop);
    hipEventSynchronize(stop);

    float ms = 0.0f;

    hipEventElapsedTime(&ms, start, stop);

    ms /= 10.0f;

    hipMemcpy(h_C, d_C, sC, hipMemcpyDeviceToHost);

    // Validation

    matmul_cpu(h_A, h_B, h_ref, M, N, K);

    float maxErr = 0.0f;

    for (int i = 0; i < M * N; i++)
        maxErr = fmax(maxErr, fabs(h_C[i] - h_ref[i]));

    double gflops =
        2.0 * M * N * K /
        (ms * 1e-3) /
        1e9;

    printf("Kernel:    COALESCED\n");

    printf(
        "Block:     %dx%d = %d threads\n",
        block.x,
        block.y,
        block.x * block.y);

    printf("Time:      %.3f ms\n", ms);

    printf("GFLOPS:    %.1f\n", gflops);

    printf("Max error: %e\n", maxErr);

    printf(
        "Status:    %s\n",
        maxErr < 1e-2 ? "PASS" : "FAIL");

    hipFree(d_A);
    hipFree(d_B);
    hipFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);
    free(h_ref);

    return 0;
}

In [ ]:
import subprocess
subprocess.run(['hipcc', '-O3', '-o', 'matmul_v1', 'matmul_v1.cpp'],
               capture_output=True, text=True, check=True)
result = subprocess.run(['./matmul_v1'], capture_output=True, text=True)
print("=== ITERATION 1 RESULT ===")
print(result.stdout)

## Iteration 2 — Tiling with LDS (Shared Memory)

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>Diagnosis:</b> without tiling, <code>A[row][k]</code> and <code>B[k][col]</code> are reloaded from global memory many times by neighboring threads — huge redundant traffic for large K.</p><p style='margin-bottom:0;'><b>Fix:</b> each workgroup cooperatively loads a <code>TILE_SIZE × TILE_SIZE</code> block of A and B into LDS, reuses it many times, then moves to the next tile (HBM → LDS → compute). Traffic drops, reuse rises, performance improves.</p>
</div>

In [ ]:
%%writefile matmul_v2.cpp

#include <hip/hip_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>


// ============================================================
// Iteration 2:
// Add tiling with LDS (shared memory)
//
// Changes from Iteration 1:
//   - Cooperative tile loading
//   - Reuse data from LDS
//   - Reduce HBM memory traffic
//
// Each workgroup loads:
//
//   TILE_SIZE x TILE_SIZE
//
// blocks of A and B into LDS,
// computes partial sums,
// then advances to the next tile.
// ============================================================

#define TILE_SIZE 16


__global__ void matmul_tiled(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    // Fast on-chip shared memory (LDS)

    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    // Global matrix coordinates

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    // Local coordinates inside the tile

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    float sum = 0.0f;

    // Process matrix tiles

    for (int t = 0; t < K; t += TILE_SIZE)
    {
        // Cooperative load from HBM into LDS

        if (row < M && (t + tx) < K)
            tileA[ty][tx] = A[row * K + (t + tx)];
        else
            tileA[ty][tx] = 0.0f;

        if (col < N && (t + ty) < K)
            tileB[ty][tx] = B[(t + ty) * N + col];
        else
            tileB[ty][tx] = 0.0f;

        // Wait for entire tile to load

        __syncthreads();

        // Compute partial sum using fast LDS memory

        for (int k = 0; k < TILE_SIZE; k++)
            sum += tileA[ty][k] * tileB[k][tx];

        // Wait before overwriting LDS tiles

        __syncthreads();
    }

    // Store final result

    if (row < M && col < N)
        C[row * N + col] = sum;
}


// CPU reference implementation

void matmul_cpu(
    const float* A,
    const float* B,
    float* C,
    int M,
    int N,
    int K)
{
    for (int row = 0; row < M; row++)
    {
        for (int col = 0; col < N; col++)
        {
            float sum = 0.0f;

            for (int k = 0; k < K; k++)
                sum += A[row * K + k] * B[k * N + col];

            C[row * N + col] = sum;
        }
    }
}


int main()
{
    int M = 1024;
    int N = 1024;
    int K = 1024;

    size_t sA = M * K * sizeof(float);
    size_t sB = K * N * sizeof(float);
    size_t sC = M * N * sizeof(float);

    float* h_A   = (float*)malloc(sA);
    float* h_B   = (float*)malloc(sB);
    float* h_C   = (float*)malloc(sC);
    float* h_ref = (float*)malloc(sC);

    srand(42);

    for (int i = 0; i < M * K; i++)
        h_A[i] = (float)rand() / RAND_MAX;

    for (int i = 0; i < K * N; i++)
        h_B[i] = (float)rand() / RAND_MAX;

    float *d_A, *d_B, *d_C;

    hipMalloc(&d_A, sA);
    hipMalloc(&d_B, sB);
    hipMalloc(&d_C, sC);

    hipMemcpy(d_A, h_A, sA, hipMemcpyHostToDevice);
    hipMemcpy(d_B, h_B, sB, hipMemcpyHostToDevice);

    // Same 16x16 workgroup as Iteration 1

    dim3 block(TILE_SIZE, TILE_SIZE);

    dim3 grid(
        (N + TILE_SIZE - 1) / TILE_SIZE,
        (M + TILE_SIZE - 1) / TILE_SIZE);

    // Warmup

    matmul_tiled<<<grid, block>>>(
        d_A, d_B, d_C,
        M, N, K);

    hipDeviceSynchronize();

    // GPU timing

    hipEvent_t start, stop;

    hipEventCreate(&start);
    hipEventCreate(&stop);

    hipEventRecord(start);

    for (int i = 0; i < 10; i++)
    {
        matmul_tiled<<<grid, block>>>(
            d_A, d_B, d_C,
            M, N, K);
    }

    hipEventRecord(stop);
    hipEventSynchronize(stop);

    float ms = 0.0f;

    hipEventElapsedTime(&ms, start, stop);

    ms /= 10.0f;

    hipMemcpy(h_C, d_C, sC, hipMemcpyDeviceToHost);

    // Validation

    matmul_cpu(
        h_A,
        h_B,
        h_ref,
        M, N, K);

    float maxErr = 0.0f;

    for (int i = 0; i < M * N; i++)
        maxErr = fmax(maxErr, fabs(h_C[i] - h_ref[i]));

    double gflops =
        2.0 * M * N * K /
        (ms * 1e-3) /
        1e9;

    printf("Kernel:    V3 (tiled + LDS)\n");

    printf(
        "Block:     %dx%d = %d threads\n",
        block.x,
        block.y,
        block.x * block.y);

    printf("Time:      %.3f ms\n", ms);

    printf("GFLOPS:    %.1f\n", gflops);

    printf("Max error: %e\n", maxErr);

    printf(
        "Status:    %s\n",
        maxErr < 1e-2 ? "PASS" : "FAIL");

    hipFree(d_A);
    hipFree(d_B);
    hipFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);
    free(h_ref);

    return 0;
}


In [ ]:
import subprocess
subprocess.run(['hipcc','-O3','-o','matmul_v2','matmul_v2.cpp'],
               capture_output=True, text=True, check=True)
result = subprocess.run(['./matmul_v2'], capture_output=True, text=True)
print("=== ITERATION 2 RESULT ===")
print(result.stdout)

## Comparing the Two Approaches

Both iterations use the **same 256-thread (16×16) workgroup** and return identical results. They fix two *different* bottlenecks, and the gains multiply.

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Approach 1 — Fix the memory access pattern (coalescing)</div>
<p>The only change from the baseline is which thread index maps to <b>row</b> vs. <b>column</b>. Inside a wavefront, <code>threadIdx.x</code> is the fast-varying lane, so it must drive the contiguous (column) dimension.</p>
</div>

```text
Baseline (uncoalesced)                   Iteration 1 (coalesced)
row = blockIdx.x*blockDim.x + tIdx.x     row = blockIdx.y*blockDim.y + tIdx.y
col = blockIdx.y*blockDim.y + tIdx.y     col = blockIdx.x*blockDim.x + tIdx.x
```

In the **baseline**, consecutive lanes get consecutive `row`, so writes to `C[row*N+col]` land N floats apart (a full 1024-element row) — every lane needs its own memory transaction. In **Iteration 1**, consecutive lanes get consecutive `col`, so reads of `B[k*N+col]` and writes of `C[row*N+col]` are contiguous and the hardware fuses them into a few wide bursts. The fix costs nothing but a reindex.

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Approach 2 — Reuse data from LDS (tiling)</div>
<p>Coalescing makes each HBM access efficient, but the kernel still re-reads the same rows of A and columns of B from HBM many times. Tiling stages them in fast on-chip shared memory (LDS) and reuses them before moving to the next slice of K.</p>
</div>

```text
for each 16-wide slice of K:
    cooperatively load a 16x16 tile of A and of B into __shared__ (LDS)
    __syncthreads()
    each thread accumulates 16 partial products from LDS
    __syncthreads()
```

Each element is now read from HBM **once per tile** and reused 16× from LDS (~100× lower latency than HBM), cutting global-memory traffic by roughly `TILE_SIZE`.

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Order matters</div>
<p>Coalescing first, then reuse. Fixing the access pattern is free and unblocks the memory system; tiling then removes the redundant traffic that remained. Production GEMM (rocBLAS) goes further with register blocking, larger/double-buffered tiles, and matrix cores.</p>
</div>

---

## Results Summary

Measured on your run (M = N = K = 1024, FP32):

| Version | Optimization | Block | Time (ms) | GFLOPS | Speedup |
|---------|--------------|-------|-----------:|--------:|---------:|
| Baseline | Naive | 16×16 (256) |      |     |    |
| Iteration 1 | Memory Coalescing | 16×16 (256) |     |    |     |
| Iteration 2 | + LDS Tiling | 16×16 (256) |     |      |     |


**Next:** Module 3 — ROCm Libraries, PyTorch & AI Frameworks